# mu2e-fastsim quickstart on EAF

The EAF path of [QUICKSTART.md](../QUICKSTART.md) as notebook cells. Fill the three values from your input-release access reply ([setup/README.md](../setup/README.md)) in the first cell and run the cells in order.

A `%%bash` cell runs in its own shell, so sourcing `site/eaf.env` there would not reach the kernel or later cells. `load_preset()` sources the preset in bash once and copies the result into `os.environ`; every later `%%bash` cell and subprocess inherits it.

In [ ]:
import os, subprocess
# from your access reply (setup/README.md); M2FS_DATA is a writable area of yours with quota, not $HOME
os.environ["M2FS_INPUTS"] = "<release root>"
os.environ["MANIFEST"] = "<manifest path>"
os.environ["M2FS_DATA"] = "<your Mu2e data area>/fastsim"
CODE = os.environ["M2FS_CODE"] = os.path.join(os.path.dirname(os.environ["M2FS_DATA"]), "mu2e-fastsim")


def load_preset():
    """Source site/eaf.env in bash and copy its environment into this kernel (os.environ)."""
    r = subprocess.run(["bash", "-c", f'cd "{CODE}" && source site/eaf.env && env -0'], capture_output=True)
    if r.returncode:
        raise RuntimeError(r.stderr.decode())
    env = dict(kv.split("=", 1) for kv in r.stdout.decode().split("\0") if "=" in kv)
    os.environ.clear(); os.environ.update(env)
    os.chdir(CODE)
    print("preset loaded:", os.environ["M2FS_CODE"], "venv on PATH:", os.environ["PATH"].startswith(os.environ["M2FS_VENV"]))

## Clone, load the preset, check the settings

In [ ]:
%%bash
set -e
mkdir -p "$M2FS_DATA"
[ -d "$M2FS_CODE" ] || git clone -q --recurse-submodules https://github.com/RobMina/mu2e-fastsim.git "$M2FS_CODE"

In [ ]:
load_preset()
print(subprocess.run(["python3", "site/check.py", "eaf"], capture_output=True, text=True).stdout)

## Python environment (once; it lives on Fermilab storage and outlives the server)

In [ ]:
%%bash
set -e
[ -x "$M2FS_VENV/bin/python" ] || /cvmfs/mu2e.opensciencegrid.org/env/ana/current/bin/python -m venv --system-site-packages "$M2FS_VENV"
"$M2FS_VENV/bin/pip" install -q "jax==0.11.2" -e core -e crv -e tracker -e beam
"$M2FS_VENV/bin/python" -c "import jax, mu2efastsim, crvfastsim, beamfastsim; print('jax', jax.__version__, jax.default_backend())"

In [ ]:
load_preset()        # again: now puts the venv first on PATH

## Verify the input release and run the demo

`check-inputs` hashes every file (about 10 s); the demo runs three short chains on the CPU (about 6 minutes on a GPVM; most of it JAX compilation and table loading) and prints each headline number next to its reference value and tolerance.

In [ ]:
%%bash
python3 -m mu2efastsim.inputs check-inputs --manifest "$MANIFEST" --root "$M2FS_INPUTS" | grep -c '"ok"'
python3 -m mu2efastsim.demo

## Run your own job

The command of the `beam_point` kit task with one changed parameter: the 49-foil target `beam/geom/move_extras12.txt` instead of the nominal 37 foils. The stop fraction should rise from about 0.36 to about 0.67.

In [ ]:
%%bash
python3 -m beamfastsim point --g4 "$M2FS_INPUTS/beam_fastsim/g4ref_full/g4ref.npz" \
  --geom beam/geom/move_extras12.txt --n 20000 --seed 1 --out "$M2FS_DATA/my_point"

In [ ]:
import json
m = json.load(open(os.path.join(os.environ["M2FS_DATA"], "my_point", "metrics.json")))
print(f"stop fraction {m['stop_frac']:.4f} +- {m['stop_frac_err']:.4f} ({m['stops']:.0f} stops of {m['muons']:.0f} muons, {m['foils']:.0f} foils)")